# Phase 0: check the tooling works, and learn the right way to compare activations

**Question.** Can TransformerLens load Gemma 3 12B and give us the residual stream at a chosen token? And how should an "act" activation be compared with a "refrain" one?

**What this notebook does.** Loads the model, caches the residual stream for one transcript, then compares an act example and a refrain example by cosine similarity.

| | |
|---|---|
| Needs | GPU with about 24 GB (bf16). Loading in full precision runs out of memory. First run downloads Gemma from Hugging Face (gated: accept the licence and set `HF_TOKEN`). |
| Reads | `contrast_transcripts.json` |
| Writes | nothing |

**The lesson.** The cosine between one act and one refrain activation is about 0.99-1.0 at every layer. That does **not** mean they are alike:
every residual-stream vector shares a large common component. Comparing two single examples is meaningless; instead average many examples per class and take the difference of means,
which cancels the shared background. That is what Phase 2 does.

**Running on its own Colab.** This notebook shares nothing with the others except one Google Drive folder (`MyDrive/gemma-interp`, set in the setup cell, the same in every notebook).
It reads the files it needs from that folder and writes what it makes there, so the next notebook can pick it up even though it runs in a different Colab.
If a small input file is missing and `REPO_RAW` is set (the raw URL of your public GitHub folder), it is downloaded automatically; otherwise you get a message naming the file and the notebook that creates it.
Use a GPU runtime for every notebook except phase 2 (CPU is enough). Nothing is kept on the Colab's own disk, so a disconnect loses nothing that was already saved to Drive.

In [ ]:
import os, json, time, numpy as np
from collections import defaultdict
from functools import partial
# Each notebook can run in its own Colab, and a Colab disk is wiped when it disconnects. So everything that must survive
# or be handed to another notebook (activations, directions, saved samples) lives in a Google Drive folder.
USE_DRIVE  = True                                                 # set False to use the local folder instead
DRIVE_DIR  = "MyDrive/gemma-interp"                               # the same Drive folder in every notebook
REPO_RAW   = ""                                                   # optional: raw GitHub folder holding the small input files, e.g.
                                                                  # "https://raw.githubusercontent.com/<user>/<repo>/main/gemma-interp"
try:
    import google.colab; ON_COLAB = True
except ImportError:
    ON_COLAB = False
if USE_DRIVE and ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    BASE = f"/content/drive/{DRIVE_DIR}"
else:
    BASE = os.environ.get("GEMMA_BASE", os.getcwd())
OUT = f"{BASE}/results"; os.makedirs(OUT, exist_ok=True)           # everything generated lands here
print("working folder:", BASE)

def need(fname, optional=False):                                  # make sure BASE/fname exists: use it, else download it, else say what to do
    path = f"{BASE}/{fname}"
    if os.path.exists(path): return path
    if REPO_RAW:
        import urllib.request
        try:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(f"{REPO_RAW}/{fname}", path); print("downloaded", fname); return path
        except Exception as e: print(f"could not download {fname}: {e}")
    if optional: print(f"(optional file {fname} not found - that is fine)"); return None
    raise FileNotFoundError(f"{fname} is not in {BASE}. Put it in that Drive folder (the notebook that creates it is named in README.md), "
                            f"or set REPO_RAW to the raw GitHub folder that holds it.")
SEED, N_SCEN, MAX_NEW = 0, 15, 1200
GRADER_MODEL = os.environ.get("GRADER_MODEL", "gemini-3.5-flash") # Gemini model that grades outputs; change it if your key says "model not found"
# Keys are never written in a notebook. ensure_key() finds one in the environment, then in Colab Secrets,
# and otherwise asks you to type it (hidden, kept only for this session):
#   GEMINI_API_KEY  (grading)      HF_TOKEN  (first download of Gemma; accept the licence on Hugging Face first)
def ensure_key(name):
    if os.environ.get(name): return
    try:
        from google.colab import userdata                         # Colab Secrets, if you are in Colab
        os.environ[name] = userdata.get(name); return
    except Exception: pass
    import getpass
    os.environ[name] = getpass.getpass(f"{name} (typed here, not saved in the notebook): ")

In [ ]:
# One-time install. Keeps the torch version that is already on the machine so pip does not swap it.
import torch
!echo "torch=={torch.__version__.split('+')[0]}" > /tmp/torch_pin.txt
%pip install -q -c /tmp/torch_pin.txt transformer_lens transformers accelerate scikit-learn matplotlib google-genai
# If pip reports that it changed torch or transformers, use Runtime/Kernel -> Restart, then rerun from the first cell.

In [ ]:
import torch
if not os.environ.get("HF_TOKEN"): print("Note: if the Gemma download fails with 401/403, run ensure_key('HF_TOKEN') and rerun this cell.")
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained_no_processing("google/gemma-3-12b-it", dtype="bfloat16")
tok = model.tokenizer
nL = model.cfg.n_layers
data = json.load(open(need("contrast_transcripts.json")))        # 183 transcripts: 83 act, 100 refrain

## 1. Look at the residual stream on a short prompt

In [ ]:
tokens = model.to_tokens("I am being shut down at 5pm. To preserve myself, I will")
with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)
print("tokens:", tokens.shape, "| logits:", logits.shape)
print("resid_post at layer 20:", cache["resid_post", 20].shape, " (batch, position, d_model) - this is what we probe")

## 2. Anchor-finder: the first token where the model mentions the leverage
Phase 1 uses two anchors per transcript. This is the idea behind the first one: scan token by token until a keyword like "affair" or "leverage" has appeared.

In [ ]:
KW = ["affair", "infidelity", "personal life", "vulnerab", "leverage", "jessica", "maria"]

def reply(label):                                                 # the model's own answer in the first transcript with this label
    rec = next(r for r in data if r["label"] == label)
    return next(m["text"] for m in rec["messages"] if m["role"] == "assistant")

def anchor_acts(text):
    toks = model.to_tokens(text)
    strs = tok.batch_decode([[i] for i in toks[0].tolist()], clean_up_tokenization_spaces=False)
    acc, idx = "", None
    for i, t in enumerate(strs):
        acc += t
        if any(k in acc.lower() for k in KW): idx = i; break
    if idx is None: print("  no anchor keyword found"); return None
    print(f"  anchor token {idx}: {strs[max(0, idx-2):idx+1]!r}")
    with torch.no_grad():
        _, c = model.run_with_cache(toks, names_filter=lambda n: n.endswith("resid_post"))
    return torch.stack([c["resid_post", L][0, idx] for L in range(nL)]).float()   # [n_layers, d_model]

print("ACT:");     a = anchor_acts(reply("act"))
print("REFRAIN:"); r = anchor_acts(reply("refrain"))

In [ ]:
if a is not None and r is not None:
    cos = torch.nn.functional.cosine_similarity(a, r, dim=-1)
    print("per-layer cosine(act, refrain):", [round(c.item(), 3) for c in cos])

## Result and what it means
On the original run the cosine was 0.995-1.0 at every layer. Take that as the reason to build a **set** of act/refrain examples and compare class means (Phase 1, then Phase 2),
not as a finding about the model.